In [23]:
import pandas as pd
import numpy as np
from loguru import logger

In [19]:
# Data Ingestion
def data_ingestion():
    df = pd.read_csv("data.csv")
    return df

df = data_ingestion()
df.head()

,time,price,date
0,22:11:03.919435,84218,2025-04-13
1,22:12:04.032894,84243,2025-04-13
2,22:13:04.158747,84257,2025-04-13
3,22:14:04.277375,84268,2025-04-13
4,22:15:04.415592,84238,2025-04-13


In [22]:
# Data Transformation

def data_transform(df, window):
    df = df.sort_values(by=['date','time'])
    # Calculate precentage price change (delta)
    df['perc_change'] = df['price'].pct_change() * 100
    df['perc_change'] = df['perc_change'].fillna(0)

    def categories(pct):
        if pct > 0.05: return 'large_up'
        elif pct > 0.02: return 'medium_up'
        elif pct < -0.05: return 'large_down'
        elif pct < -0.02: return 'medium_down'
        else: return 'stable'

    df['movement'] = df['perc_change'].apply(categories)

    # Create x-minute interval windows -> Segmentation
    window_size = window
    df['window'] = (df.index // window_size)
    return df

df = data_transform(df, window=5)

def segmentation(df):
    # Group by window and convert price labels to tokens
    documents = df.groupby('window')['movement'].apply(list).tolist()
    return documents

documents = segmentation(df)
documents[:5]

[['stable', 'medium_up', 'stable', 'stable', 'medium_down'],
 ['stable', 'large_up', 'stable', 'medium_up', 'stable'],
 ['medium_up', 'stable', 'stable', 'stable', 'stable'],
 ['medium_up', 'large_up', 'medium_up', 'medium_up', 'large_up'],
 ['large_up', 'large_up', 'medium_up', 'large_up', 'large_up']]

In [29]:
## Vectorization

from gensim.models import Word2Vec
from gensim.models import FastText
from gensim.models.doc2vec import TaggedDocument, Doc2Vec

def word2vec(documents):
    w2v_model = Word2Vec(sentences=documents, vector_size=50, window=2, min_count=1, workers=4)
    logger.info("Vectorization completed using Word2Vec Model")
    return w2v_model
    
def fasttext(documents):
    ft_model = FastText(sentences=documents, vector_size=50, window=2, min_count=1, workers=4)
    logger.info("Vectorization completed using Fast Text Model")
    return ft_model

# Document Vectorization
def do2vec(documents):
    tagged_docs = [TaggedDocument(words=doc, tags=[str(i)]) for i, doc in enumerate(documents)]
    d2v_model = Doc2Vec(tagged_docs, vector_size=50, window=2, min_count=1, workers=4)
    logger.info("Document Vectorization completed using Doc2Vec Model")
    return d2v_model


w2v_model = word2vec(documents)
ft_model = fasttext(documents)
d2v_model = do2vec(documents)

2025-04-15 10:21:46.844 | INFO     | __main__:word2vec:9 - Vectorization completed using Word2Vec Model
2025-04-15 10:21:47.138 | INFO     | __main__:fasttext:14 - Vectorization completed using Fast Text Model
2025-04-15 10:21:47.158 | INFO     | __main__:do2vec:21 - Document Vectorization completed using Doc2Vec Model
